# Autoencoder for Dimensionality Reduction

This notebook demonstrates using a **simple autoencoder** (neural network) to reduce 3D data down to 2D — an alternative to PCA for non-linear dimensionality reduction.

### Pipeline

1. **Synthetic Data** — Generate 300 samples with 2 informative features (two clusters) + 1 noise dimension using `make_blobs`
2. **3D Visualization** — Plot all 3 features to see the cluster structure embedded in noise
3. **Autoencoder** — Train a 3→2→3 network (encoder compresses to 2D bottleneck, decoder reconstructs)
4. **2D Projection** — Use the encoder half to extract the learned 2D representation and verify clusters are preserved

### Key Concepts

* **Bottleneck architecture** — The 2-neuron hidden layer forces the network to learn the most informative 2D compression
* **MSE reconstruction loss** — The autoencoder learns to minimize information loss during compression
* **vs PCA** — Autoencoders can capture non-linear structure that PCA misses (though for this simple example, both work)

In [0]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs

In [0]:
# Generate 300 samples in 2 clusters (2 informative features)
data = make_blobs(n_samples=300, n_features=2, centers=2, cluster_std=1.0, random_state=101)

X, y = data

# Add a 3rd feature: pure Gaussian noise (uninformative dimension)
np.random.seed(seed=101)
z_noise = np.random.normal(size=len(X))
z_noise = pd.Series(z_noise)

# Combine into a 3-column DataFrame: X1, X2 (signal) + X3 (noise)
feat = pd.DataFrame(X)
feat = pd.concat([feat, z_noise], axis=1)
feat.columns = ['X1', 'X2', 'X3']

# 2D scatter shows the two clusters clearly (ignoring noise dimension)
plt.scatter(feat['X1'], feat['X2'], c=y)
plt.xlabel('X1')
plt.ylabel('X2')
plt.title('2D View: Two Clusters')
plt.show()

In [0]:
# 3D scatter to visualize the full feature space (X1, X2 are clustered; X3 is noise)
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(feat['X1'], feat['X2'], feat['X3'], c=y)
ax.set_xlabel('X1')
ax.set_ylabel('X2')
ax.set_zlabel('X3 (noise)')
ax.set_title('3D Feature Space')
plt.show()

In [0]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from sklearn.preprocessing import MinMaxScaler

# Architecture: 3 inputs --> 2 hidden neurons (bottleneck) --> 3 outputs
# The encoder compresses 3D to 2D; the decoder reconstructs back to 3D

encoder = Sequential()
encoder.add(Dense(units=2, activation='relu', input_shape=[3]))

decoder = Sequential()
decoder.add(Dense(units=3, activation='relu', input_shape=[2]))

autoencoder = Sequential([encoder, decoder])
autoencoder.compile(optimizer=SGD(learning_rate=1.5), loss='mse')

# Scale features to [0, 1] for stable training
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(feat)

# Train: input = output (reconstruct the original data through the bottleneck)
autoencoder.fit(scaled_data, scaled_data, epochs=5)